# Week2_ex3 - Coil Around a Straight Wire (H-field)

This exercise looks at the H-field made by a long straight wire and checks how much of it links through two ring coils placed at different distances along the wire, using the EddyCurrent solver in Maxwell 3D.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil

In [ ]:
# 1. open Ansys Electronics Desktop (AEDT)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create project and set solution type
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. get odesign object so recorded GUI scripts can be pasted directly
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##
proj_name = "Week2_ex3"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)
desi_name = "Week2_ex3"

## Save project and apply design name ##
proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)
M3D.save_project()


In [ ]:
# function that finds the terminal faces of the coil
# uses the center coordinate of each face, finds the face with the largest absolute x position, and returns the two faces
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position of winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # append terminal face to array
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
line_length = 400
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
# 1. setup code copy-pasted from GUI script recording
oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 10,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 15,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"CacheSaveKind:="	, "Delta",
		"ConstantDelta:="	, "0s",
		"UseCacheFor:="		, ["Freq"],
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, True,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])

# 2. store setup name only (not the pyaedt setup object) --
# AEDT 2025 R2 renamed the EddyCurrent solver to "AC Magnetic" internally,
# and pyaedt 0.15.3's M3D.setups[-1] lookup doesn't recognize the new name (KeyError: 'ac magnetic').
# so we just keep the setup name as a string and call the raw AEDT API later instead of the pyaedt wrapper.
setup_name = "Setup1"

In [ ]:
# 1. draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                                    xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


# 2. create region and set radiation boundary condition
region_tmp = ["line_length/2", "-line_length/2", "line_length*2", "-line_length*2", "line_length*2", "-line_length*2"]
region = M3D.modeler.create_region(pad_value=region_tmp ,pad_type="Absolute Position")

assignment = [region.top_face_y, region.bottom_face_y, region.top_face_z, region.bottom_face_z]

# NOTE: M3D.assign_radiation() checks that solution_type == "EddyCurrent" exactly and
# raises AEDTRuntimeError("Excitation applicable only to Eddy Current.") otherwise. But
# AEDT 2025 R2 renamed this solver "AC Magnetic" internally, so self.solution_type reports
# "AC Magnetic" here -- the check fails even though this IS an eddy current solve. Bypassing
# the pyaedt wrapper and calling the raw AEDT boundary API directly sidesteps this check.
oModule = oDesign.GetModule("BoundarySetup")
face_ids = [f.id for f in assignment]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])


# 3. create dummy object
box_origin = ["25mm", "-line_length*2", "-line_length/2"]
box_sizes = ["-2*25mm", "2*line_length", "line_length"]
dummy = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes,name="dummy", material="vacuum")
M3D.modeler.subtract(blank_list=dummy, tool_list=line, keep_originals=True)


# 4. draw ring coil
center = [0, "-line_length*(3/4)", "-1mm"]
origin = [0, "-line_length*(3/4)-20mm", "-1mm"]
ring_1 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name="ring_1", material="copper")

center = [0, "-line_length*(3/4)", "-1mm"]
origin = [0, "-line_length*(3/4)-18mm", "-1mm"]
tmp = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name=None, material=None)
M3D.modeler.subtract(blank_list=ring_1, tool_list=tmp, keep_originals=False)

M3D.modeler.subtract(blank_list=dummy, tool_list=ring_1, keep_originals=True)


# 5. create surface on ring coil face to mark the region for data extraction
origin = [0, "-line_length*(3/4)", 0]
field_circle_1 = M3D.modeler.create_circle(orientation="XY", origin=origin, radius="20mm", 
                                           num_sides=0, is_covered=True, name="field_circle_1", material=None, non_model=False)


# 6. repeat steps 4-5 to create ring_2, field_circle_2
center = [0, "-line_length*(1/4)", "-1mm"]
origin = [0, "-line_length*(1/4)-20mm", "-1mm"]
ring_2 = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name="ring_2", material="copper")

center = [0, "-line_length*(1/4)", "-1mm"]
origin = [0, "-line_length*(1/4)-18mm", "-1mm"]
tmp = M3D.modeler.create_polyhedron(orientation="Z", center=center, origin=origin, height="2mm", num_sides=36, name=None, material=None)
M3D.modeler.subtract(blank_list=ring_2, tool_list=tmp, keep_originals=False)

M3D.modeler.subtract(blank_list=dummy, tool_list=ring_2, keep_originals=True)

origin = [0, "-line_length*(1/4)", 0]
field_circle_2 = M3D.modeler.create_circle(orientation="XY", origin=origin, radius="20mm", 
                                           num_sides=0, is_covered=True, name="field_circle_2", material=None, non_model=False)


In [ ]:
oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:dummy_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "3000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/8",
	])

In [ ]:
# 1. set the terminal faces of the wire object as coil
ter_out, ter_in = find_terminal_face(line)  # call function to find terminal faces of the coil

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


# 2. create winding and add the coils to it
coil = M3D.assign_winding(assignment=None, winding_type='Current', is_solid=True, current="Current", 
                        resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                        name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

oModule.AddNamedExpression("coil1_H", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_H"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle_1"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Mean"]
	])
	
oModule.AddNamedExpression("coil2_H", "Fields", 
	[
		"NameOfExpression:="	, ["Vector_H"],
		"Operation:="		, ["Normal"],
		"Operation:="		, ["Dot"],
		"EnterSurface:="	, ["field_circle_2"],
		"Operation:="		, ["SurfaceValue"],
		"Operation:="		, ["Mean"]
	])

In [ ]:
# my_setup.analyze() doesn't work here for the same "AC Magnetic" naming reason, so call
# the raw AEDT API with the setup name string instead.
oDesign.Analyze(setup_name)

In [ ]:
oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Calculator Expressions Table 1", "Fields", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"Freq:="		, ["All"],
		"Phase:="		, ["0deg"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Freq",
		"Y Component:="		, ["coil1_H","coil2_H"]
	])


In [ ]:
oModule = oDesign.GetModule("FieldsReporter")

oModule.CreateFieldPlot(
	[
		"NAME:Mag_B1",
		"SolutionName:="	, "Setup1 : LastAdaptive",
		"UserSpecifyName:="	, 0,
		"UserSpecifyFolder:="	, 0,
		"QuantityName:="	, "Mag_B",
		"PlotFolder:="		, "B",
		"StreamlinePlot:="	, False,
		"AdjacentSidePlot:="	, False,
		"FullModelPlot:="	, False,
		"IntrinsicVar:="	, "Freq=\'10Hz\' Phase=\'0deg\'",
		"PlotGeomInfo:="	, [1,"Volume","ObjList",1,"dummy"],
		"FilterBoxes:="		, [0],
		[
			"NAME:PlotOnVolumeSettings",
			"PlotIsoSurface:="	, True,
			"PointSize:="		, 1,
			"Refinement:="		, 0,
			"CloudSpacing:="	, 0.5,
			"CloudMinSpacing:="	, -1,
			"CloudMaxSpacing:="	, -1,
			"ShadingType:="		, 0,
			"IsoMapTransparency:="	, True,
			"IsoTransparency:="	, 0.899999976158142,
			"IsoTransScaleThreshold:=", 0.200000002980232,
			[
				"NAME:Arrow3DSpacingSettings",
				"ArrowUniform:="	, True,
				"ArrowSpacing:="	, 0,
				"MinArrowSpacing:="	, 0,
				"MaxArrowSpacing:="	, 0
			]
		],
		"EnableGaussianSmoothing:=", False,
		"SurfaceOnly:="		, False
	], "Field")

oModule.SetPlotFolderSettings("B", 
	[
		"NAME:FieldsPlotSettings",
		"Real time mode:="	, True,
		[
			"NAME:ColorMapSettings",
			"ColorMapType:="	, "Spectrum",
			"SpectrumType:="	, "Rainbow",
			"UniformColor:="	, [127,255,255],
			"RampColor:="		, [255,127,127]
		],
		[
			"NAME:Scale3DSettings",
			"unit:="		, 104,
			"m_nLevels:="		, 10,
			"minvalue:="		, 1.9635E-05,
			"maxvalue:="		, 0.00025,
			"log:="			, False,
			"IntrinsicMin:="	, 1.96351322052549E-05,
			"IntrinsicMax:="	, 0.00438627410447518,
			"LimitFieldValuePrecision:=", False,
			"FieldValuePrecisionDigits:=", 4,
			"dB:="			, False,
			"AnimationStaticScale:=", False,
			"ScaleType:="		, 1,
			"UserSpecifyValues:="	, [11,1.96350002288818E-05,0.000456298919677734,0.000892962829589844,0.00132962670898437,0.00176629064941406,0.00220295458984375,0.00263961840820312,0.00307628247070312,0.0035129462890625,0.00394961010742187,0.00438627392578125],
			"ValueNumberFormatTypeAuto:=", 0,
			"ValueNumberFormatTypeScientific:=", False,
			"ValueNumberFormatWidth:=", 8,
			"ValueNumberFormatPrecision:=", 3
		],
		[
			"NAME:Marker3DSettings",
			"MarkerType:="		, 0,
			"MarkerMapSize:="	, False,
			"MarkerMapColor:="	, False,
			"MarkerSize:="		, 0.25
		],
		[
			"NAME:Arrow3DSettings",
			"ArrowType:="		, 1,
			"ArrowMapSize:="	, False,
			"ArrowMapColor:="	, True,
			"ShowArrowTail:="	, True,
			"ArrowSize:="		, 0.100000001490116,
			"ArrowMinMagnitude:="	, -0.499980364867795,
			"ArrowMaxMagnitude:="	, 0.504386274104475,
			"ArrowMagnitudeThreshold:=", 0,
			"ArrowMagnitudeFilteringFlag:=", False,
			"ArrowMinIntrinsicMagnitude:=", 0,
			"ArrowMaxIntrinsicMagnitude:=", 1
		],
		[
			"NAME:DeformScaleSettings",
			"ShowDeformation:="	, True,
			"MinScaleFactor:="	, 0,
			"MaxScaleFactor:="	, 1,
			"DeformationScale:="	, 0,
			"ShowDeformationOutline:=", False
		]
	])

In [ ]:
M3D.save_project()   
